# Main
Runs the deletion-audit pipeline for every active table in the control table, then advances each table's watermark.

In [0]:
%run ./00_config

In [0]:
%run ./01_core_functions

## Load active tables to process

In [0]:
control_df = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("is_active") == 1)
)

In [0]:
control_rows = control_df.collect()

## Process each table and advance its watermark
Each table is processed and its `last_processed_version` updated in the same iteration

In [0]:
for row in control_rows:

    catalog_name = row["catalog_name"]
    schema_name = row["schema_name"]
    table_name = row["table_name"]

    primary_key_columns = (
        row["primary_key_columns"]
    )

    last_processed_version = (
        row["last_processed_version"]
    )

    # just in case
    if last_processed_version is None:
        last_processed_version = 1

    try:

        processed_version = process_table(
            catalog_name,
            schema_name,
            table_name,
            primary_key_columns,
            last_processed_version
        )
       
        # Update control table
        spark.sql(f"""
            UPDATE {CONTROL_TABLE}
            SET
                last_processed_version = {processed_version},
                updated_at = current_timestamp()
            WHERE
                catalog_name = '{catalog_name}'
                AND schema_name = '{schema_name}'
                AND table_name = '{table_name}'
        """)


        print(
            f"Updated watermark for "
            f"{catalog_name}.{schema_name}.{table_name} "
            f"to version {processed_version}"
        )


    except Exception as e:

        print(
            f"FAILED: "
            f"{catalog_name}.{schema_name}.{table_name}"
        )

        print(str(e))

        raise

## Inspect the audit log

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_test ORDER BY table_name, commit_version, primary_id

In [0]:
%sql
INSERT INTO moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL_TEST
(
    catalog_name,
    schema_name,
    table_name,
    capture_columns,
    last_processed_version,
    is_active,
    updated_at
)
VALUES
(
    'iics',
    'casemix',
    'patient_details_test',
    'customer_id',
    1,
    TRUE,
    current_timestamp()
);

In [0]:
%sql
DROP TABLE IF EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_TEST;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_TEST
(
    audit_id          STRING,
    catalog_name      STRING,
    schema_name       STRING,
    table_name        STRING,
    primary_id        STRING,
    deleted_by        STRING,
    deleted_at        TIMESTAMP,
    commit_version    BIGINT,
    commit_timestamp  TIMESTAMP,
    operation         STRING,
    loaded_at         TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DROP TABLE IF EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL_TEST;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL_TEST
(
    catalog_name           STRING,
    schema_name            STRING,
    table_name             STRING,
    capture_columns        STRING,
    last_processed_version BIGINT,
    is_active              BOOLEAN,
    updated_at             TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  moh_raz.raz_casemix_stg.casemix.patient_details_test
(
    customer_id BIGINT,
    name        STRING,
    dob         DATE,
    status      STRING
)
USING DELTA;

In [0]:
%sql
SHOW PROCEDURES IN moh_raz.raz_casemix_stg

In [0]:
%sql
DESCRIBE PROCEDURE moh_raz.raz_casemix_stg.sp_capture_deletions;

In [0]:
%sql
DESCRIBE PROCEDURE EXTENDED moh_raz.raz_casemix_stg.sp_capture_deletions;

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL_TEST

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_TEST

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_stg.patient_details_test

In [0]:
%sql
-- Create table with CDF enabled
DROP TABLE IF EXISTS moh_raz.raz_casemix_transformed.patient_test;

In [0]:
%sql
-- Create table with CDF enabled
CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_transformed.patient_test (
    healthcare_facility STRING,
    patient_account_number STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);

In [0]:
%sql

-- Insert sample data
INSERT INTO moh_raz.raz_casemix_transformed.patient_test
    (healthcare_facility, patient_account_number)
VALUES
    ('HF001', 'PA10001'),
    ('HF002', 'PA10002'),
    ('HF003', 'PA10003'),
    ('HF001', 'PA10004'),
    ('HF002', 'PA10005');

In [0]:
%sql
-- Create table with CDF enabled
DROP TABLE IF EXISTS moh_raz.raz_casemix_transformed.patient_test_1;

In [0]:
%sql
-- Create table with CDF enabled
CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_transformed.patient_test_1 (
    healthcare_f STRING,
    patient_acc_no STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);

In [0]:
%sql

-- Insert sample data
INSERT INTO moh_raz.raz_casemix_transformed.patient_test_1
    (healthcare_f, patient_acc_no)
VALUES
    ('HF100', 'PA20001'),
    ('HF102', 'PA20002'),
    ('HF103', 'PA20003'),
    ('HF101', 'PA20004'),
    ('HF102', 'PA20005');

In [0]:
%sql
delete from moh_raz.raz_casemix_transformed.patient_test where healthcare_facility='HF001'

In [0]:
%sql
delete from moh_raz.raz_casemix_transformed.patient_test_1 where healthcare_f='HF101'

In [0]:
%sql
select * from moh_raz.raz_casemix_transformed.css_deletion_lookup

In [0]:
%sql
DROP TABLE  IF EXISTS moh_raz.raz_casemix_transformed.css_deletion_aud_control;
DROP TABLE  IF EXISTS moh_raz.raz_casemix_transformed.t_ft_cs_deletion_aud;
DROP TABLE  IF EXISTS moh_raz.raz_casemix_stg.css_deletion_audit;

In [0]:
%sql
select * from moh_raz.raz_casemix_transformed.css_deletion_aud_control

In [0]:
%sql
select * from moh_raz.raz_casemix_transformed.t_ft_cs_deletion_aud

In [0]:
%sql
select * from moh_raz.raz_casemix_stg.css_deletion_audit

In [0]:
%sql
CREATE TABLE IF NOT EXISTS moh_raz.raz_casemix_transformed.css_deletion_lookup (
    s_no     INT,
    catalog  STRING,
    schema   STRING,
    `table`  STRING,
    columns  STRING,
    is_active INT
)
USING DELTA;

INSERT INTO moh_raz.raz_casemix_transformed.css_deletion_lookup
    (s_no, catalog, schema, `table`, columns, is_active)
VALUES
    (1, 'moh_raz', 'raz_casemix_transformed', 'patient_test',
     'healthcare_facility,patient_account_number', 1),

    (2, 'moh_raz', 'raz_casemix_transformed', 'patient_test_1',
     'healthcare_f,patient_acc_no', 1);